## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

## Dataset

In [10]:
from datasets.Student_t import MultivariateStudentT

d = 64
dim_x = dim_y = d//2
rho = 0.7
df = 1
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  12.46
X torch.Size([10000, 32]) Y torch.Size([10000, 32])


## Hyperaparams

In [11]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.K_components = 6
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [12]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.000566381961107254 loss val= 0.008904023095965385 best val loss= 0.008904023095965385 best t= 0
finished: t= 76 loss= -0.023459501564502716 loss val= 0.08745865523815155 best val loss= -0.007036566734313965 best t= 22
finished: t= 152 loss= 0.6119977235794067 loss val= 0.5694711208343506 best val loss= -0.06275878846645355 best t= 150
finished: t= 228 loss= -0.05658829212188721 loss val= -0.03523952513933182 best val loss= -0.06275878846645355 best t= 150
finished: t= 304 loss= -0.0712965726852417 loss val= 0.1077924519777298 best val loss= -0.07949073612689972 best t= 273
finished: t= 380 loss= -0.2532455027103424 loss val= -0.13823577761650085 best val loss= -0.13823577761650085 best t= 380
finished: t= 456 loss= -0.17887204885482788 loss val= 0.32311397790908813 best val loss= -0.15371039509773254 best t= 382
finished: t= 532 loss= -0.39394140243530273 loss val= -0.0579352080821991 best val loss= -0.18286430835723877 best t= 487
finished: t= 608 loss= -0.41235

In [13]:
## Copula neural density estimate
from estimators.CODINE import CODINE

estimator = CODINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.01890123263001442 loss val= 0.01908569410443306 best val loss= 0.01908569410443306 best t= 0
finished: t= 76 loss= -5.593453407287598 loss val= -5.244889259338379 best val loss= -5.5660834312438965 best t= 65
finished: t= 152 loss= -7.164112091064453 loss val= -5.553431510925293 best val loss= -5.8915557861328125 best t= 120
finished: t= 228 loss= -5.609775543212891 loss val= -5.5923566818237305 best val loss= -5.926527976989746 best t= 194
finished: t= 304 loss= -5.884455680847168 loss val= -5.512603759765625 best val loss= -6.040070533752441 best t= 299
finished: t= 380 loss= -6.198564529418945 loss val= -5.669336318969727 best val loss= -6.040070533752441 best t= 299
finished: t= 456 loss= -5.618870735168457 loss val= -5.647951126098633 best val loss= -6.040070533752441 best t= 299


true MI: 12.456365411743121
est MI: 6.9742255210876465


In [15]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


K components= 6 copula transform= True
nde type: FM
finished: t= 0 loss= 3.5052506923675537 loss val= 2.306431770324707 best val loss= 2.306431770324707 best t= 0
finished: t= 126 loss= 0.22863003611564636 loss val= 1.3763360977172852 best val loss= 1.370041847229004 best t= 22
finished: t= 252 loss= 0.14168402552604675 loss val= 1.272207498550415 best val loss= 1.2183388471603394 best t= 226
finished: t= 378 loss= 0.15300503373146057 loss val= 1.342064380645752 best val loss= 1.2183388471603394 best t= 226
finished: t= 504 loss= 0.21938863396644592 loss val= 1.2991435527801514 best val loss= 1.0941486358642578 best t= 475
finished: t= 630 loss= 0.17084287106990814 loss val= 1.4676889181137085 best val loss= 1.0430476665496826 best t= 553


finished: t= 0 loss= 1.0368623733520508 loss val= 2.1767592430114746 best val loss= 2.1767592430114746 best t= 0
finished: t= 126 loss= 0.3035867512226105 loss val= 1.7017192840576172 best val loss= 1.3915247917175293 best t= 106
finished: t= 252 lo

In [14]:
## CopulaMI
from estimators.CopulaMI import CopulaMI

estimator = CopulaMI(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

true MI: 12.456365411743121
est MI: 8.426759719848633
